Dado que el entrenamiento de redes neuronales es una tarea  muy costosa, **se recomienda ejecutar el notebooks en [Google Colab](https://colab.research.google.com)**, por supuesto también se puede ejecutar en local.

Al entrar en [Google Colab](https://colab.research.google.com) bastará con hacer click en `upload` y subir este notebook. No olvide luego descargarlo en `File->Download .ipynb`

**El examen deberá ser entregado con las celdas ejecutadas, si alguna celda no está ejecutadas no se contará.**

El examen se divide en tres partes, con la puntuación que se indica a continuación. La puntuación máxima será 10.

    
- [Actividad 1: Redes Recurrentes](#actividad_1): 10 pts
    - [Cuestión 1](#3.1): 2.5 pt
    - [Cuestión 2](#3.2): 2.5 pt
    - [Cuestión 3](#3.3): 2.5 pts
    - [Cuestión 4](#3.4): 1.25 pts
    - [Cuestión 5](#3.5): 1.25 pts



In [34]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

<a name='actividad_1'></a>
# Actividad 1: Redes Recurrentes


- [Cuestión 1](#3.1): 2.5 pt
- [Cuestión 2](#3.2): 2.5 pt
- [Cuestión 3](#3.3): 2.5 pts
- [Cuestión 4](#3.4): 1.25 pts
- [Cuestión 5](#3.5): 1.25 pts

Vamos a usar un dataset de las temperaturas mínimas diarias en Melbourne. La tarea será la de predecir la temperatura mínima en dos días. Puedes usar técnicas de series temporales vistas en otras asignaturas, pero no es necesario.


In [35]:
dataset_url = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/daily-min-temperatures.csv'
data_dir = tf.keras.utils.get_file('daily-min-temperatures.csv', origin=dataset_url)

In [36]:
df = pd.read_csv(data_dir, parse_dates=['Date'])
df.head()

,Date,Temp
0,1981-01-01,20.7
1,1981-01-02,17.9
2,1981-01-03,18.8
3,1981-01-04,14.6
4,1981-01-05,15.8


In [37]:
temperatures = df['Temp'].values
print('number of samples:', len(temperatures))
train_data = temperatures[:3000]
test_data = temperatures[3000:]
print('number of train samples:', len(train_data))
print('number of test samples:', len(test_data))
print('firsts trainn samples:', train_data[:10])

number of samples: 3650
number of train samples: 3000
number of test samples: 650
firsts trainn samples: [20.7 17.9 18.8 14.6 15.8 15.8 15.8 17.4 21.8 20. ]


<a name='3.1'></a>
## Cuestión 1: Convierta `train_data` y `test_data`  en ventanas de tamaño 5, para predecir el valor en 2 días

En la nomenclatura de [Introduction_to_RNN_Time_Series.ipynb](https://github.com/ezponda/intro_deep_learning/blob/main/class/RNN/Introduction_to_RNN_Time_Series.ipynb)
```python
past, future = (5, 2)
```

Para las primeras 10 muestras de train_data `[20.7, 17.9, 18.8, 14.6, 15.8, 15.8, 15.8, 17.4, 21.8, 20. ]` el resultado debería ser:

```python
x[0] : [20.7, 17.9, 18.8, 14.6, 15.8] , y[0]: 15.8
x[1] : [17.9, 18.8, 14.6, 15.8, 15.8] , y[1]: 17.4
x[2] : [18.8, 14.6, 15.8, 15.8, 15.8] , y[2]: 21.8
x[3] : [14.6, 15.8, 15.8, 15.8, 17.4] , y[3]: 20.             
```

In [38]:
def create_windows_tf(data, window_size, horizon, shuffle=True):
    """
    Crea un dataset con los datos secuenciales dados usando tf.data.Dataset

    Parametros:
    ----------
    data (np.array): Time series en una dimensión.
    window_size (int): Número de valores pasados que se usarán como entrada.
    horizon: Número de valores que se van a predecir.
    shuffle (bool): Aleatoriedad.

    Devuelve:
    --------
    tf.data.Dataset: el dataset de resultado
    """

    ts_data = tf.data.Dataset.from_tensor_slices(data)
    ts_data = ts_data.window(window_size + horizon, shift=1, drop_remainder=True)
    ts_data = ts_data.flat_map(lambda window: window.batch(window_size+horizon))
    ts_data = ts_data.map(lambda window: (window[:window_size], window[window_size:]))

    if shuffle:
        ts_data = ts_data.shuffle(buffer_size=data.shape[0])

    X, y = [], []
    for x_batch, y_batch in ts_data:
        X.append(x_batch.numpy())
        y.append(y_batch.numpy())
    return np.array(X), np.array(y)


In [39]:
past, future = 5, 2
X_train, y_train = create_windows_tf(train_data, past, future, shuffle=True)
X_test, y_test = create_windows_tf(test_data, past, future, shuffle=True)

<a name='3.2'></a>
## Cuestión 2: Cree un modelo recurrente de dos capas GRU para predecir con las ventanas de la cuestión anterior.


In [40]:
inputs = keras.layers.Input(shape=(past, 1))

GRU_1 = keras.layers.GRU(20, return_sequences=True)(inputs)
GRU_2 = keras.layers.GRU(20, return_sequences=False)(GRU_1)

outputs = keras.layers.Dense(future)(GRU_2)

model = keras.Model(inputs=inputs, outputs=outputs)
model.compile(
    optimizer=keras.optimizers.Adam(),
    loss="mse"
)
model.summary()

Model: "functional_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)      │ (None, 5, 1)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_12 (GRU)                    │ (None, 5, 20)          │         1,380 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_13 (GRU)                    │ (None, 20)             │         2,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 2)              │            42 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,942 (15.40 KB)

 Trainable params: 3,942 (15.40 KB)

 Non-trainable params: 0 (0.00 B)

In [41]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 5s 27ms/step - loss: 124.5891 - val_loss: 93.8521
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 80.8372 - val_loss: 66.0255
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 61.1587 - val_loss: 51.0602
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 48.0802 - val_loss: 42.2483
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 40.2140 - val_loss: 35.9197
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 34.5605 - val_loss: 31.0688
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 29.4505 - val_loss: 27.3206
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 26.9283 - val_loss: 24.4711
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 23.7198 - val_loss: 22.3498
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 22.2162 - val_loss: 20.7535
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - loss: 21.4560 - val_loss: 19.5969
Epoch 12/200
38/38 ━━━━━━━━━━

In [42]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.9925
Test Loss: 5.964301109313965


<a name='3.3'></a>
## Cuestión 3: Añada más features a la series temporal, por ejemplo `portion_year`. Cree un modelo que mejore al anterior.


In [43]:
## Puede añadir más features
df['portion_year'] = df['Date'].dt.dayofyear / 365.0
df_multi = df[['Temp', 'portion_year']].copy()

## train - test split
train_data = df_multi.iloc[:3000].copy()
test_data = df_multi.loc[3000:, :].copy()

In [45]:
## Create windows
def create_windows_multivariate_numpy(data, window_size, horizon, target_col_index=0, shuffle=True):
    """
    Crea ventanas de una serie temporal multivariable y devuelve X, y como arrays NumPy.

    Parámetros:
    -----------
    data : np.array o DataFrame con forma (timesteps, features)
    window_size : int, número de pasos pasados
    horizon : int, número de pasos futuros
    target_col_index : int, índice de la columna objetivo
    shuffle : bool, si se deben mezclar las ventanas

    Devuelve:
    --------
    X : np.array de forma (num_muestras, window_size, num_features)
    y : np.array de forma (num_muestras, horizon)
    """
    data = np.array(data, dtype=np.float32)
    X, y = [], []

    for i in range(len(data) - window_size - horizon + 1):
        x_window = data[i:i + window_size]
        y_window = data[i + window_size:i + window_size + horizon, target_col_index]
        X.append(x_window)
        y.append(y_window)

    X = np.array(X)
    y = np.array(y)

    if shuffle:
        indices = np.arange(len(X))
        np.random.shuffle(indices)
        X = X[indices]
        y = y[indices]

    return X, y

past, future = 5, 2

X_train, y_train = create_windows_multivariate_numpy(train_data, past, future, target_col_index=0)
X_test, y_test = create_windows_multivariate_numpy(test_data, past, future, target_col_index=0)

In [47]:
inputs = keras.layers.Input(shape=(past, 2))

x = keras.layers.GRU(20, return_sequences=True)(inputs)
x = keras.layers.GRU(20)(x)

outputs = keras.layers.Dense(future)(x)

model = keras.Model(inputs=inputs, outputs=outputs)

model.compile(
    optimizer=keras.optimizers.Adam(),
    loss='mse'
)

model.summary()

Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 5, 2)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_14 (GRU)                    │ (None, 5, 20)          │         1,440 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_15 (GRU)                    │ (None, 20)             │         2,520 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 2)              │            42 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,002 (15.63 KB)

 Trainable params: 4,002 (15.63 KB)

 Non-trainable params: 0 (0.00 B)

In [48]:
es_callback = keras.callbacks.EarlyStopping(
    monitor="val_loss", min_delta=0, patience=10)

history = model.fit(
    X_train, y_train,
    epochs=200,
    validation_split=0.2, shuffle=True, batch_size = 64, callbacks=[es_callback]
)

Epoch 1/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 6s 46ms/step - loss: 136.2957 - val_loss: 102.7979
Epoch 2/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - loss: 91.4608 - val_loss: 68.8773
Epoch 3/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - loss: 62.4621 - val_loss: 55.5087
Epoch 4/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 51.0103 - val_loss: 46.4338
Epoch 5/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 44.5048 - val_loss: 39.3945
Epoch 6/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 37.7745 - val_loss: 33.9008
Epoch 7/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 31.9621 - val_loss: 29.6793
Epoch 8/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 29.7828 - val_loss: 26.3972
Epoch 9/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - loss: 24.5284 - val_loss: 23.9506
Epoch 10/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 23.5156 - val_loss: 22.0722
Epoch 11/200
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 21.4854 - val_loss: 20.6539
Epoch 12/200
38/38 ━━━━━━━━━

In [49]:
results = model.evaluate(X_test, y_test, verbose=1)
print('Test Loss: {}'.format(results))

21/21 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.0489
Test Loss: 5.40435266494751


<a name='3.4'></a>
## Cuestión 4: ¿En cuáles de estas aplicaciones se usaría un arquitectura 'many-to-one'?

**a)** Clasificación de sentimiento en textos

**b)** Verificación de voz para iniciar el ordenador.

**c)** Generación de música.

**d)** Un clasificador que clasifique piezas de música según su autor.


a, b, d

<a name='3.5'></a>
## Cuestión 5: ¿Qué ventajas aporta el uso de word embeddings?

**a)** Permiten reducir la dimensión de entrada respecto al one-hot encoding.

**b)** Permiten descubrir la similaridad entre palabras de manera más intuitiva que con one-hot encoding.

**c)** Son una manera de realizar transfer learning en nlp.

**d)** Permiten visualizar las relaciones entre palabras con métodos de reducción de dimensioones como el PCA.


a, b, c, d